In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/tejaswinina/transliteration-output/romanized_vocab.csv
/kaggle/input/datasets/tejaswinina/transliteration-output/__huggingface_repos__.json
/kaggle/input/datasets/tejaswinina/transliteration-output/xlit_cache.csv
/kaggle/input/datasets/tejaswinina/transliteration-output/pipeline.py
/kaggle/input/datasets/tejaswinina/final-emotion-dataset/Noisy.csv
/kaggle/input/datasets/tejaswinina/final-emotion-dataset/Synset.csv
/kaggle/input/datasets/tejaswinina/final-emotion-dataset/Synonyms.csv
/kaggle/input/datasets/tejaswinina/final-emotion-dataset/Gold_dataset.csv


In [2]:
# ================================================================
#  CELL 1 — INSTALL
#
#  RUN THIS ON THE PYTHON 3.10 IMAGE (fork the public IndicXlit notebook
#  with Copy & Edit). On 3.10 the real AI4Bharat model installs and NB0 is
#  not needed at all — this notebook builds the transliteration cache
#  itself and saves it for the other paper's run to reuse.
#
#  On the 3.12 image everything still runs, but the indicxlit arms are
#  skipped and you get the itrans arms only.
#
#  Settings -> Internet: ON
#  Add Input: emotion-dataset  (+ optionally another run's xlit_cache.csv)
# ================================================================
import subprocess, sys, os, shutil
def sh(c):
    print("+", c); return subprocess.run(c, shell=True).returncode

sh('pip install -q regex emoji wordfreq indic-nlp-library aksharamukha indic-transliteration')

PY310 = sys.version_info[:2] == (3, 10)
print("\npython:", sys.version.split()[0], "->",
      "3.10 image: IndicXlit available" if PY310 else "3.12 image: ITRANS only")
if PY310:
    sh('pip install -q ai4bharat-transliteration')
    try:
        from ai4bharat.transliteration import XlitEngine
        print("ai4bharat-transliteration: OK")
    except Exception as e:
        print("ai4bharat import failed:", str(e)[:160])

for p in ("pipeline.py", "/kaggle/working/pipeline.py"):
    if os.path.exists(p): os.remove(p); print("removed stale", p)
shutil.rmtree("__pycache__", ignore_errors=True)
sys.modules.pop("pipeline", None)


+ pip install -q regex emoji wordfreq indic-nlp-library aksharamukha indic-transliteration
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 30.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.3/40.3 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 289.9/289.9 kB 12.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.9/162.9 kB 10.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 9.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 48.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 16.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.7/7.7 MB 92.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 38.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.1/121.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 

In [3]:
%%writefile pipeline.py
"""
pipeline.py v5.3 — shared method for BOTH papers.

Cell 0 and Cell 1 are byte-identical in NB1_paper1 and NB1_paper2.
Only the final cell differs, and only in WHICH ARMS it selects.

Changes from the two original notebooks:
  * split is keyed on the id column of Gold_dataset.csv, computed BEFORE
    anything is fitted, identical in both papers
  * transliteration uses AI4Bharat IndicXlit (neural), not sanscript/ITRANS
  * LCCN uses the improved guard: G-test on label distributions,
    Dirichlet-smoothed evidence, cluster coherence, orthographic guard,
    on top of the original emotion / negation / length vetoes
  * SEGMENT is one shared setting (it was ON in paper 2, OFF in paper 1)
  * label column resolves to majority / gold_label / label
"""
PIPELINE_VERSION = "5.3"

import os, re, glob, math, hashlib, warnings
import numpy as np, pandas as pd, regex, emoji
from collections import Counter, defaultdict
warnings.filterwarnings("ignore")

SEED   = 42
WORK   = "/kaggle/working"
N_VAL  = N_TEST = 6000
LABELSET = {"happy", "sad", "anger", "disgust"}

# ---- shared decisions: these apply to BOTH papers ----
SEGMENT         = True    # paper 2 used True, paper 1 used False. Pick once.

# ---- which transliteration engine backs the transliteration arms ----
#   "itrans"    : indic_transliteration.sanscript, rule-based ITRANS scheme
#                 conversion. Installs anywhere, runs in seconds. Arms are
#                 named itrans / keel_itrans so the columns say what was run.
#   "indicxlit" : AI4Bharat IndicXlit, the neural model. Needs a cache built
#                 outside Kaggle (build_xlit_cache.py) because fairseq does
#                 not install on Python 3.12. Arms are named indicxlit /
#                 keel_indicxlit.
# The arm NAMES follow this setting, so a results table can never claim
# IndicXlit while ITRANS was actually run.
XLIT_BACKEND    = "both"   # "itrans" | "indicxlit" | "both"
                           # "both" builds itrans/keel_itrans always (free) and
                           # adds indicxlit/keel_indicxlit when xlit_cache.csv
                           # is attached, so one run yields both backends on
                           # identical splits and they can be compared directly.
DIRICHLET_ALPHA = 0.5     # Jeffreys prior for label evidence
MAX_EDIT        = 2       # orthographic guard on the de-duplication fallback
G_THRESHOLD     = None    # None = chi-square p=0.01 critical value for the dof

os.makedirs(f"{WORK}/splits", exist_ok=True)
np.random.seed(SEED)

def find(kw):
    hits = [f for f in glob.glob("/kaggle/input/**/*.csv", recursive=True)
            if kw.lower() in os.path.basename(f).lower()]
    assert hits, f"No CSV contains '{kw}'"
    return hits[0]

def load_lc(p):
    d = pd.read_csv(p, dtype=str, keep_default_na=False)
    d.columns = [c.strip().lower().lstrip("\ufeff") for c in d.columns]
    return d

def is_kn(s):  return any('\u0c80' <= ch <= '\u0cff' for ch in str(s))

# ============================================================ corpus audit
def load_corpus(path=None):
    df = load_lc(path or find("Gold_dataset"))
    LAB = next((c for c in ("majority","gold_label","label","emotion") if c in df.columns), None)
    assert LAB, f"no label column in {list(df.columns)}"
    TXT = next((c for c in ("comment","comments","text") if c in df.columns), None)
    ID  = next((c for c in ("id","comment_id","cid") if c in df.columns), None)
    RATERS = [c for c in ("ae1","ae2","ae3") if c in df.columns]

    keep = [c for c in ([ID] if ID else []) + [TXT] + RATERS + [LAB] if c]
    df = df[keep].copy()
    # Case is noise in this corpus: allcaps (28,678) and titlecase (28,635) are
    # the second and third most common noisetypes in the lexicon. Fold it once,
    # here, so every arm including raw sees the same casing. Folding it only
    # inside the lexicon arms would make their change rate reflect casing
    # rather than normalization.
    df[TXT] = (df[TXT].str.replace(r"\s+", " ", regex=True).str.strip().str.lower())
    for c in RATERS + [LAB]:
        df[c] = df[c].str.strip().str.lower()
    df = df[(df[TXT] != "") & (df[LAB].isin(LABELSET))]

    if ID is None:
        df["id"] = df[TXT].map(lambda t: hashlib.sha1(t.encode()).hexdigest()[:12])
        ID = "id"
        print("no id column -> derived id = sha1(comment)[:12]")
    assert df[ID].is_unique, "id column is not unique"

    n0 = len(df)
    df = df.drop_duplicates(subset=[TXT, LAB])                       # exact duplicates
    conf = df.groupby(TXT)[LAB].nunique()
    df = df[~df[TXT].isin(conf[conf > 1].index)]                     # label conflicts
    df = df.sort_values(ID).reset_index(drop=True)
    print(f"corpus: {n0:,} -> {len(df):,} after audit (dedup + conflict removal)")
    print(df[LAB].value_counts().to_string())
    return df, ID, TXT, LAB, RATERS

# ================================================================ tokenizer
from wordfreq import zipf_frequency
def is_english(w): return zipf_frequency(w.lower(), "en") >= 2.5

KAN      = regex.compile(r'^\p{Script=Kannada}+$')
TOKEN_RE = regex.compile(r'https?://\S+|@\w+|#\w+|\p{Script=Kannada}+|[A-Za-z]+|\d+(?:\.\d+)?|\X',
                         regex.UNICODE)
_ELONG   = regex.compile(r'(.)\1{2,}')
_DEDUP   = regex.compile(r'(.)\1+')
VOWELS   = set("aeiou")

def trim_elongation(w): return _ELONG.sub(r'\1\1', w)

SEG_VOCAB = set()          # filled by build_keel()

def segment_latin(tok, max_pieces=4):
    low = tok.lower()
    if low in SEG_VOCAB or is_english(low) or len(low) < 6: return [tok]
    pieces, i, n = [], 0, len(low)
    while i < n:
        hit = None
        for j in range(n, i, -1):
            if low[i:j] in SEG_VOCAB:
                hit = j; pieces.append(low[i:j]); i = j; break
        if hit is None: return [tok]
        if len(pieces) > max_pieces: return [tok]
    return pieces if len(pieces) > 1 else [tok]

def tag_token(t):
    if t.startswith("http"): return "url"
    if t.startswith("@"):    return "mention"
    if t.startswith("#"):    return "hashtag"
    if KAN.match(t):         return "kannada"
    if t.isdigit() or re.match(r'^\d+\.\d+$', t): return "number"
    if t in emoji.EMOJI_DATA or any(c in emoji.EMOJI_DATA for c in t): return "emoji"
    if t.isalpha(): return "english" if is_english(t) else "romanized_kannada"
    return "punctuation"

PASSTHROUGH = ("emoji", "number", "punctuation", "url", "mention", "hashtag")

def process_comment(text, segment=None):
    seg = SEGMENT if segment is None else segment
    raw = [t for t in TOKEN_RE.findall(str(text)) if t.strip()]
    toks = []
    for t in raw:
        if t.isalpha() or KAN.match(t): t = trim_elongation(t)
        if seg and t.isalpha() and t.isascii(): toks.extend(segment_latin(t))
        else: toks.append(t)
    return toks, [tag_token(t) for t in toks]

# ================================ KEEL lexicon: three-tier cascade
class KeelLexicon:
    """Resolves a token against the lexicon in three tiers, in order:

      TIER 1  Noisy.csv      the token is a known misspelling
                             -> canonical form of its va_id, emotion of its synset
      TIER 2  Synonyms.csv   the token is already a listed variant
                             -> canonical form of its va_id, emotion of its synset
      TIER 3  Synset.csv     the token is a canonical word with no generated
                             variants (203 synsets are Synset-only)
                             -> unchanged, emotion of its synset
      MISS                   not in the lexicon at all -> unchanged, no emotion.
                             The LCCN guard then falls back to LabelEvidence,
                             which is learned from the training split.

    Within a tier the token is tried exact, lowercased, elongation-trimmed,
    then fully de-duplicated, so the same cascade applies at every tier.
    Emotion is stored as a SET: a spelling listed under two synsets carries no
    single reliable emotion and is exempt from the emotion veto.
    """
    SINGLE_TOKEN_ONLY = True   # the tokenizer splits on whitespace, so
                               # multi-word entries can never match. Keeping
                               # them would invert deletions such as
                               # drop_stopword ("happy" -> "be happy").

    def __init__(self, syn, noise, base, unify_script=False):
        self.sid2emo = dict(zip(base["synset_id"], base["emotion"]))
        self.unify = unify_script
        va2 = {}
        for va, w, t, sid in zip(syn["va_id"], syn["word"],
                                 syn["transliteration"], syn["synset_id"]):
            va2[va] = (str(w).strip(), str(t).strip(), sid)

        def canon_for(va, want_kn):
            """Canonical form for a va_id. Script-preserving by default so the
            arm does not silently transliterate; unify_script=True collapses
            both scripts of a va_id onto one form."""
            w, t, sid = va2[va]
            if self.unify:
                return (w if is_kn(w) else t) or w or t
            if want_kn:  return w if is_kn(w) else (t if is_kn(t) else None)
            return w if (w and not is_kn(w)) else (t if (t and not is_kn(t)) else None)

        self.T1, self.T2, self.T3 = {}, {}, {}
        self.emo = defaultdict(set)
        self.phrase_emo = defaultdict(set)   # negated phrases, for the veto
        miss_va = dropped_multi = 0
        one = lambda s: len(str(s).split()) == 1

        # ---- TIER 1: noisy forms ----
        cnt = defaultdict(Counter)
        for va, nf, sid in zip(noise["va_id"], noise["noisy_form"], noise["synset_id"]):
            if va not in va2: miss_va += 1; continue
            nf = str(nf).strip()
            if not nf: continue
            low = nf.lower()
            e = self.sid2emo.get(sid)
            c = canon_for(va, is_kn(nf))
            if self.SINGLE_TOKEN_ONLY and not (one(nf) and c and one(c)):
                dropped_multi += 1
                if e and c and not one(c) and str(c).split()[0].lower() in NEG:
                    self.phrase_emo[str(c).strip().lower()].add(e)
                continue
            if e: self.emo[low].add(e)
            if c and c.lower() != low: cnt[low][c.lower()] += 1
        self.T1 = {k: v.most_common(1)[0][0] for k, v in cnt.items()}

        # ---- TIER 2: synonym surface forms ----
        for va, (w, t, sid) in va2.items():
            e = self.sid2emo.get(sid)
            for surf in (w, t):
                if not surf: continue
                low = surf.strip().lower()
                if not one(surf):
                    # multi-word: unusable for word-level lookup, but a negated
                    # phrase is exactly what the context-aware veto needs
                    if e and low.split()[0] in NEG: self.phrase_emo[low].add(e)
                    continue
                if e: self.emo[low].add(e)
                c = canon_for(va, is_kn(surf))
                if c and c.lower() != low and one(c) and low not in self.T1:
                    self.T2.setdefault(low, c.lower())

        # ---- TIER 3: canonical words with no variants ----
        cols = [c for c in ("canonical_word", "english_word") if c in base.columns]
        for col in cols:
            for f, sid in zip(base[col], base["synset_id"]):
                low = str(f).strip().lower()
                if not low: continue
                e = self.sid2emo.get(sid)
                if not one(low):
                    if e and low.split()[0] in NEG: self.phrase_emo[low].add(e)
                    continue
                if e: self.emo[low].add(e)
                if low not in self.T1 and low not in self.T2:
                    self.T3[low] = low

        self.emo = dict(self.emo); self.phrase_emo = dict(self.phrase_emo)
        self.stats = Counter()
        n_amb = sum(1 for v in self.emo.values() if len(v) > 1)
        print(f"  KEEL cascade | tier1 {len(self.T1):,} | tier2 {len(self.T2):,} "
              f"| tier3 {len(self.T3):,} | emotions {len(self.emo):,} "
              f"({n_amb:,} ambiguous)")
        print(f"    multi-word entries skipped (word-level lookup): {dropped_multi:,}")
        print(f"    negated phrases kept for the context-aware veto: "
              f"{len(self.phrase_emo):,} | va_id misses {miss_va}")

    def emotion_in_context(self, tok, prev=None):
        """'not happy' is sad while 'happy' alone is happy. When the previous
        token is a negator, the two-token phrase is looked up first."""
        low = tok.lower()
        if prev and prev.lower() in NEG:
            e = self.phrase_emo.get(f"{prev.lower()} {low}")
            if e: return e
        return self.emo.get(low, set())

    def _variants(self, w):
        low = w.lower()
        out = [w, low]
        if w.isalpha() or is_kn(w):
            tr = trim_elongation(low)
            if tr != low: out.append(tr)
            dd = _DEDUP.sub(r"\1", low)
            if dd != low and orthographically_plausible(low, dd): out.append(dd)
        seen = set()
        return [v for v in out if not (v in seen or seen.add(v))]

    def resolve(self, w):
        """Returns (canonical_form, emotions:set, tier) with tier in
        {1, 2, 3, 0} where 0 means the token is not in the lexicon."""
        for v in self._variants(w):
            for tier, table in ((1, self.T1), (2, self.T2), (3, self.T3)):
                if v in table:
                    self.stats[f"tier{tier}"] += 1
                    return table[v], self.emo.get(v, set()), tier
        self.stats["miss"] += 1
        low = w.lower()
        base_form = trim_elongation(low) if (w.isalpha() or is_kn(w)) else w
        return base_form, self.emo.get(low, set()), 0

    def normalize(self, w):
        return self.resolve(w)[0]

    def emotion(self, w):
        return self.resolve(w)[1]

    def report(self):
        t = sum(self.stats.values())
        if not t: return
        print("  lexicon resolution over the corpus:")
        for k in ("tier1", "tier2", "tier3", "miss"):
            n = self.stats[k]
            label = {"tier1": "tier 1 (noisy form)", "tier2": "tier 2 (synonym)",
                     "tier3": "tier 3 (synset only)", "miss": "not in lexicon"}[k]
            print(f"    {label:<24}{n:>10,}  {100*n/t:5.2f}%")

def build_keel(unify_script=False):
    global SEG_VOCAB
    syn   = load_lc(find("Synonyms"))
    noise = load_lc(find("Noisy"))
    base  = load_lc(find("Synset"))

    lex = KeelLexicon(syn, noise, base, unify_script=unify_script)

    # flat map kept for the arms that only need noisy -> canonical
    norm_map = dict(lex.T1); norm_map.update(lex.T2)
    multi = {k: v for k, v in norm_map.items() if len(str(v).split()) > 1}
    for k, v in multi.items(): norm_map[k] = "".join(str(v).split())
    if multi:
        print(f"  {len(multi):,} entries had a multi-word canonical form; "
              f"whitespace collapsed to keep mappings one-to-one")
    norm_map = {k: v for k, v in norm_map.items() if k and v}

    SEG_VOCAB = {str(w).strip().lower() for w in syn["word"]
                 if str(w).strip().isalpha() and str(w).strip().isascii()
                 and len(str(w).strip()) >= 3}

    F2E = lex.emo
    print(f"  KEEL entries: {len(norm_map):,} | seg vocab: {len(SEG_VOCAB):,} "
          f"| form->emotion: {len(F2E):,}")
    return norm_map, F2E, lex

NEG = {"illa","ila","alla","ala","beda","bedha","bekagilla","gottilla","gotilla","ilva",
       "illave","aagalla","agalla","maadalla","madalla","saadhyavilla","sadyavilla",
       "not","no","never","dont","cant","wont"}

# ================================================== phonetic key functions
_SX = {**{c:"1" for c in "bfpv"}, **{c:"2" for c in "cgjkqsxz"},
       **{c:"3" for c in "dt"}, "l":"4", **{c:"5" for c in "mn"}, "r":"6"}
def soundex(w):
    w = re.sub(r"[^a-z]", "", str(w).lower())
    if not w: return ""
    o, p = w[0], _SX.get(w[0], "")
    for ch in w[1:]:
        dd = _SX.get(ch, "")
        if dd and dd != p: o += dd
        if ch not in "hw": p = dd
    return (o + "000")[:4]

_IND = [(r"ksh","x"),(r"gy","gn"),(r"chh","ch"),(r"bh","b"),(r"dh","d"),(r"gh","g"),(r"jh","j"),
        (r"kh","k"),(r"ph","f"),(r"th","t"),(r"w","v"),(r"q","k"),(r"z","j"),
        (r"aa","a"),(r"ee","i"),(r"ii","i"),(r"oo","u"),(r"uu","u")]
def indic_soundex(w):
    w = str(w).lower()
    for p, r in _IND: w = re.sub(p, r, w)
    return soundex(w)

_MT = [(r"^(kn|gn|pn|ae|wr)", lambda m: m.group(0)[1]),(r"^x","s"),(r"^wh","w"),(r"mb$","m"),
       (r"sch","sk"),(r"tch","ch"),(r"ck","k"),(r"ph","f"),(r"gh","f"),(r"th","0"),(r"sh","x"),
       (r"ch","x"),(r"c([iey])",r"s\1"),(r"c","k"),(r"d?g([iey])",r"j\1"),(r"g","k"),(r"q","k"),
       (r"v","f"),(r"z","s"),(r"([bcdfghjklmnpqrstvwxyz0])\1+",r"\1")]
def metaphone(w):
    w = re.sub(r"[^a-z]", "", str(w).lower())
    if not w: return ""
    for p, r in _MT: w = re.sub(p, r, w)
    return (w[0] + re.sub(r"[aeiou]", "", w[1:]))[:8]

_TR = [(r"aa|ee|ii|oo|uu",""),(r"kh|gh|q","k"),(r"bh|ph|f","p"),(r"dh|th","t"),(r"jh|zh|z","j"),
       (r"sh|s","s"),(r"ch","c"),(r"w","v"),(r"y","i"),(r"x","ks")]
def terun(w):
    w = re.sub(r"[^a-z]", "", str(w).lower())
    if not w: return ""
    h = w[0]
    for p, r in _TR: w = re.sub(p, r, w)
    return re.sub(r"(.)\1+", r"\1", h + (re.sub(r"[aeiou]", "", w[1:]) if len(w) > 1 else ""))

_K = [(r"aa","a"),(r"ee","i"),(r"oo","u"),(r"dh","d"),(r"bh","b"),(r"ph","p"),(r"kh","k"),(r"gh","g")]
def kope(w):
    w = re.sub(r"(.)\1{2,}", r"\1\1", str(w)).lower()
    for p, r in _K: w = re.sub(p, r, w)
    return w
def kope_ok(t): return t.isalpha() and t.isascii() and not is_english(t)

# ======================================== improved LCCN guard components
class LabelEvidence:
    """Dirichlet-smoothed label evidence per surface form, with a G-test for
    whether two forms carry significantly different evidence. Replaces the
    argmax comparison and the hard MIN_SUPPORT cutoff."""
    CRIT = {1:6.63, 2:9.21, 3:11.34, 4:13.28, 5:15.09, 6:16.81}      # chi2, p=0.01
    def __init__(self, frame, TXT, LAB, alpha=DIRICHLET_ALPHA):
        self.alpha = alpha
        self.counts = defaultdict(Counter)
        self.labels = sorted(frame[LAB].unique()); self.K = len(self.labels)
        for txt, lab in zip(frame[TXT], frame[LAB]):
            for t in set(process_comment(txt)[0]):
                self.counts[t.lower()][lab] += 1
        tot = Counter()
        for c in self.counts.values(): tot.update(c)
        n = max(sum(tot.values()), 1)
        self.prior = np.array([tot[l]/n for l in self.labels])
    def vec(self, f):
        c = self.counts.get(f)
        return np.zeros(self.K) if not c else np.array([c[l] for l in self.labels], float)
    def posterior(self, f):
        v = self.vec(f); a = self.alpha * self.K * self.prior
        return (v + a) / (v.sum() + a.sum())
    def g_test(self, f1, f2):
        o = np.vstack([self.vec(f1), self.vec(f2)]); n = o.sum()
        if n == 0: return 0.0, self.K - 1
        e = np.outer(o.sum(1), o.sum(0)) / n
        m = (o > 0) & (e > 0)
        G = 2.0 * np.sum(o[m] * np.log(o[m] / e[m]))
        dof = (o.shape[0]-1) * (np.count_nonzero(o.sum(0)) - 1)
        return float(G), max(dof, 1)
    def conflict(self, f1, f2, threshold=G_THRESHOLD):
        G, dof = self.g_test(f1, f2)
        t = threshold if threshold is not None else self.CRIT.get(dof, 6.63 + 2.6*dof)
        return G > t
    def jsd(self, f1, f2):
        p, q = self.posterior(f1), self.posterior(f2); m = 0.5*(p+q)
        kl = lambda a, b: float(np.sum(a * np.log2(a/b)))
        return 0.5*kl(p, m) + 0.5*kl(q, m)

class Clusters:
    """Union-find, so a merge is judged against the whole equivalence class
    it would join, not just one canonical form."""
    def __init__(self): self.parent = {}; self.members = defaultdict(set)
    def find(self, x):
        self.parent.setdefault(x, x)
        while self.parent[x] != x:
            self.parent[x] = self.parent[self.parent[x]]; x = self.parent[x]
        return x
    def cluster_of(self, x):
        r = self.find(x); return self.members[r] | {r}
    def union(self, a, b):
        ra, rb = self.find(a), self.find(b)
        if ra == rb: return
        self.parent[ra] = rb
        self.members[rb] |= self.members[ra] | {ra}
        self.members.pop(ra, None)

def edit_distance(a, b, cap=3):
    if abs(len(a)-len(b)) > cap: return cap+1
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[j-1]+1, prev[j-1]+(ca != cb)))
        prev = cur
        if min(prev) > cap: return cap+1
    return prev[-1]

def skeleton(w): return "".join(c for c in w.lower() if c not in VOWELS)

def orthographically_plausible(v, c, max_edit=MAX_EDIT):
    """Blocks boss -> bos, sulle -> sule and similar spurious collapses."""
    if v == c: return True
    if skeleton(v) != skeleton(c): return False
    return edit_distance(v, c, cap=max_edit) <= max_edit

def make_veto(F2E, lex=None):
    """Three vetoes: emotion clash, negation flip, length gap.

    The emotion veto fires only when BOTH forms carry exactly one emotion and
    those differ. A form listed under two synsets carries no reliable single
    emotion, so it falls through to the G-test, which uses corpus evidence
    rather than an arbitrary row order in the lexicon files."""
    def veto(w, c, prev=None):
        if lex is not None:
            a, b = lex.emotion_in_context(w, prev), lex.emotion_in_context(c, prev)
        else:
            a, b = F2E.get(w), F2E.get(c)
        if a and b and len(a) == 1 and len(b) == 1 and a != b: return "emotion"
        if (w in NEG) != (c in NEG): return "negation"
        if abs(len(w) - len(c)) > 3: return "length"
        return None
    return veto

class LCCN:
    """Label-constrained cascade normalization.
    Original vetoes plus G-test, cluster coherence and orthographic guard."""
    def __init__(self, norm_map, evidence, veto, translit=None,
                 use_gtest=True, use_cluster=True, use_ortho=True, use_veto=True,
                 lex=None):
        self.map = norm_map or {}; self.ev = evidence; self.veto = veto
        self.lex = lex
        self.translit = translit
        self.use_gtest, self.use_cluster = use_gtest, use_cluster
        self.use_ortho, self.use_veto = use_ortho, use_veto
        self.clusters = Clusters(); self.cache = {}; self.stats = Counter()

    def candidates(self, tok):
        low = tok.lower(); out = []
        if self.lex is not None:
            c, _e, tier = self.lex.resolve(tok)
            if c and c != low: out.append((c, f"tier{tier}"))
        for cand, tier in ((low, "lower"), (trim_elongation(low), "trim"),
                           (_DEDUP.sub(r"\1", low), "dedup")):
            if cand in self.map: out.append((self.map[cand], tier))
        if kope_ok(tok):
            kw = kope(low)
            if kw and kw != low: out.append((kw, "kope"))
        return out

    def choose(self, tok, prev=None):
        low = tok.lower(); best, best_j = low, float("inf")
        for cand, tier in self.candidates(tok):
            if cand == low: continue
            if self.use_ortho and tier in ("dedup","kope") \
               and not orthographically_plausible(low, cand):
                self.stats["blocked_orthographic"] += 1; continue
            if self.use_veto:
                g = self.veto(low, cand, prev)
                if g: self.stats["blocked_veto_" + g] += 1; continue
            if self.use_gtest and self.ev.conflict(low, cand):
                self.stats["blocked_gtest"] += 1; continue
            if self.use_cluster and any(self.ev.conflict(low, m)
                                        for m in self.clusters.cluster_of(cand)):
                self.stats["blocked_cluster"] += 1; continue
            j = self.ev.jsd(low, cand)
            if j < best_j: best, best_j = cand, j
        if best != low: self.clusters.union(low, best); self.stats["merged"] += 1
        else: self.stats["kept"] += 1
        return best

    def resolve(self, tok, prev=None):
        neg = bool(prev and prev.lower() in NEG)
        key = (neg, tok.lower())
        if key not in self.cache: self.cache[key] = self.choose(tok, prev)
        return self.cache[key]

    def normalize(self, text):
        toks, tags = process_comment(text); out = []
        for i, (tok, tag) in enumerate(zip(toks, tags)):
            if tag in PASSTHROUGH: out.append(tok); continue
            w = self.resolve(tok, toks[i-1] if i else None)
            if self.translit and tag == "romanized_kannada": w = self.translit(w)
            out.append(w)
        return " ".join(out)

    def report(self, name="lccn"):
        t = self.stats["merged"] + self.stats["kept"]
        print(f"  {name}: merged {self.stats['merged']:,} of {t:,} distinct forms")
        for k, v in sorted(self.stats.items()):
            if k.startswith("blocked"): print(f"    {k:<28}{v:>8,}")
        return dict(self.stats)

# ====================================== AI4Bharat IndicXlit, via a cache
#
# fairseq (which ai4bharat-transliteration depends on) does not build on
# Python 3.12, and Kaggle now runs 3.12. Transliteration is a pure per-word
# lookup over roughly 27k unique romanized types, so it is computed ONCE
# outside Kaggle with build_xlit_cache.py and read here from a CSV.
#
#   1. run CELL 2 -> it writes romanized_vocab.csv and stops
#   2. run build_xlit_cache.py on that file, on Python 3.10 or 3.11
#   3. upload xlit_cache.csv as a Kaggle dataset, attach it, rerun
#
# This is also better for the paper: the mapping is fixed, inspectable and
# releasable, instead of depending on a model that may not install later.

class XlitCache:
    """roman -> Kannada lookup produced by AI4Bharat IndicXlit."""
    def __init__(self, path):
        m = pd.read_csv(path, dtype=str, keep_default_na=False)
        m.columns = [c.strip().lower() for c in m.columns]
        a, b = m.columns[0], m.columns[1]
        self.cache = dict(zip(m[a].str.strip().str.lower(), m[b].str.strip()))
        ok = sum(1 for v in self.cache.values() if is_kn(v))
        print(f"IndicXlit cache: {len(self.cache):,} entries "
              f"({100*ok/max(len(self.cache),1):.1f}% to Kannada script)")
    def word(self, w):
        return self.cache.get(w.lower(), w)
    def coverage(self, types):
        types = set(types)
        hit = len(types & set(self.cache))
        print(f"cache covers {hit:,} of {len(types):,} romanized types "
              f"({100*hit/max(len(types),1):.1f}%)")
        return hit / max(len(types), 1)

class ItransXlit:
    """Rule-based roman -> Kannada via indic_transliteration.sanscript.

    This is scheme conversion, not a learned model: it maps ITRANS spellings
    mechanically, so vowel length is only right when the writer spelled it
    the ITRANS way. Report it as ITRANS transliteration and cite the library,
    never the IndicXlit paper.
    """
    name = "itrans"
    def __init__(self):
        from indic_transliteration import sanscript
        from indic_transliteration.sanscript import transliterate
        self._s, self._t = sanscript, transliterate
        self.cache = {}
        print("transliteration backend: ITRANS (indic_transliteration.sanscript)")
    def word(self, w):
        if w not in self.cache:
            try:
                self.cache[w] = self._t(w, self._s.ITRANS, self._s.KANNADA)
            except Exception:
                self.cache[w] = w
        return self.cache[w]
    def coverage(self, types):
        types = set(types)
        ok = sum(1 for w in types if is_kn(self.word(w)))
        print(f"converted {ok:,} of {len(types):,} romanized types to Kannada "
              f"script ({100*ok/max(len(types),1):.1f}%)")
        return ok / max(len(types), 1)


def romanized_types(df, TXT):
    v = set()
    for txt in df[TXT]:
        for t, k in zip(*process_comment(txt)):
            if k == "romanized_kannada": v.add(t.lower())
    return v

def load_xlit(df, TXT, out_dir=None, build_live=True):
    """Returns {name: engine} for every transliteration backend available.

    itrans     always available (rule-based scheme conversion)
    indicxlit  in order of preference:
                 1. an attached xlit_cache.csv
                 2. built here with the real AI4Bharat model, if
                    ai4bharat-transliteration imports (Python 3.10 image only)
                 3. skipped, with the vocabulary written out for a later build
    A cache built here is saved so the second paper's run reuses it.
    """
    out_dir = out_dir or WORK
    types = romanized_types(df, TXT)
    eng = {}

    if XLIT_BACKEND in ("itrans", "both"):
        x = ItransXlit(); x.coverage(types); eng["itrans"] = x

    if XLIT_BACKEND in ("indicxlit", "both"):
        hits = [f for f in glob.glob("/kaggle/input/**/*.csv", recursive=True)
                if "xlit_cache" in os.path.basename(f).lower()]
        if hits:
            x = XlitCache(hits[0]); x.coverage(types); eng["indicxlit"] = x
        elif build_live and _indicxlit_available():
            path = build_xlit_cache_live(sorted(types), f"{out_dir}/xlit_cache.csv")
            x = XlitCache(path); x.coverage(types); eng["indicxlit"] = x
        else:
            path = f"{out_dir}/romanized_vocab.csv"
            pd.Series(sorted(types), name="word").to_csv(path, index=False)
            print("\n" + "!"*66)
            print("indicxlit arms SKIPPED: no cache attached and the model cannot")
            print("be imported here (needs the Python 3.10 image).")
            print(f"Wrote {len(types):,} romanized types to {path}")
            print("!"*66)

    if not eng:
        raise SystemExit("no transliteration backend available")
    print(f"\ntransliteration backends ready: {list(eng)}")
    return eng

def _indicxlit_available():
    try:
        import ai4bharat.transliteration  # noqa: F401
        return True
    except Exception:
        return False

def build_xlit_cache_live(words, out_path, lang="kn", beam=10, log_every=500):
    """Transliterate the romanized vocabulary with the real AI4Bharat model.
    Only runs on the Python 3.10 image, where fairseq installs. Resumes from
    an existing cache and checkpoints as it goes."""
    import time
    from ai4bharat.transliteration import XlitEngine
    done = {}
    if os.path.exists(out_path):
        prev = pd.read_csv(out_path, dtype=str, keep_default_na=False)
        done = dict(zip(prev.iloc[:, 0], prev.iloc[:, 1]))
        print(f"resuming cache: {len(done):,} entries already present")
    todo = [w for w in words if w not in done]
    if not todo:
        return out_path

    print(f"building the IndicXlit cache for {len(todo):,} types "
          f"(lang={lang}, beam={beam}, rescore=True)")
    engine = XlitEngine(lang, beam_width=beam, rescore=True)

    def one(w):
        try:
            r = engine.translit_word(w, lang_code=lang, topk=1)
        except TypeError:
            r = engine.translit_word(w, topk=1)
        if isinstance(r, dict):
            r = r.get(lang) or next(iter(r.values()))
        return r[0] if isinstance(r, (list, tuple)) and r else (r if isinstance(r, str) else w)

    fails, t0 = 0, time.time()
    for i, w in enumerate(todo, 1):
        try:
            done[w] = one(w)
        except Exception as e:
            done[w] = w; fails += 1
            if fails <= 3: print(f"  FAILED {w!r}: {type(e).__name__} {e}")
        if i % log_every == 0:
            el = time.time() - t0
            print(f"  {i:,}/{len(todo):,}  elapsed {el/60:.1f}m  "
                  f"eta {(el/i*(len(todo)-i))/60:.1f}m  failures {fails}", flush=True)
            pd.DataFrame({"word": list(done), "kannada": list(done.values())}) \
              .to_csv(out_path, index=False)
    pd.DataFrame({"word": list(done), "kannada": list(done.values())}).to_csv(out_path, index=False)
    print(f"cache written: {out_path} ({len(done):,} entries, {fails:,} failures)")
    print("  attach this notebook's output to the other paper's NB1 to skip the rebuild")
    return out_path

def available_xlit_arms(eng, base=("{x}", "keel_{x}", "lccn_{x}", "lccn_base_{x}")):
    """Arm names for whichever backends loaded."""
    return {b.format(x=k) for k in eng for b in base}


# ================================================================ id split
def make_split(df, ID, LAB):
    """Stratified split keyed on the id column, computed BEFORE anything is
    fitted. Ordering by id makes the partition independent of file order, so
    both papers get the same train/val/test."""
    from sklearn.model_selection import train_test_split
    df = df.sort_values(ID).reset_index(drop=True)
    ids = df[[ID, LAB]]
    tr_i, tmp = train_test_split(ids, test_size=N_VAL+N_TEST, stratify=ids[LAB], random_state=SEED)
    va_i, te_i = train_test_split(tmp, test_size=N_TEST, stratify=tmp[LAB], random_state=SEED)
    S = {n: set(p[ID]) for n, p in [("train",tr_i),("val",va_i),("test",te_i)]}
    assert not (S["train"] & S["val"]) and not (S["train"] & S["test"]) \
           and not (S["val"] & S["test"]), "id leakage"
    for n, s in S.items():
        pd.Series(sorted(s), name=ID).to_csv(f"{WORK}/splits/{n}_ids.csv", index=False)
        print(f"{n:<6}{len(s):>7,}")
    h = hashlib.sha256("".join(sorted(map(str, S['test']))).encode()).hexdigest()[:16]
    print(f"leakage check passed | test-set id hash: {h}")
    return S, h

# ================================================================== arms
def build_arms(norm_map, F2E, evidence, xlit, which, LEX=None):
    """xlit is the dict returned by load_xlit(): {backend_name: engine}."""
    """Returns only the arms named in `which`. Every arm consumes the same
    token stream, so token counts stay aligned across arms."""
    veto = make_veto(F2E, LEX)

    def _indicnlp(w):
        try: return _indic_norm.normalize(w)
        except Exception: return w
    def _aksh(w):
        if is_kn(w):
            try:
                from aksharamukha import transliterate as ak
                return ak.process("Kannada", "ISO", w)
            except Exception: return w
        return w
    XL = xlit if isinstance(xlit, dict) else ({"itrans": xlit} if xlit else {})
    def _tl(backend):
        e = XL.get(backend)
        return (lambda w: e.word(w)) if e else (lambda w: w)

    def keel_lookup(w):
        """Three-tier cascade: noisy form -> synonym -> synset-only, then a
        pass-through when the token is not in the lexicon at all. The LCCN
        arms use the same cascade, so any difference between keel and lccn is
        the guards and not the lookup."""
        if LEX is not None:
            return LEX.normalize(w)
        if not norm_map:
            return trim_elongation(w.lower()) if (w.isalpha() or is_kn(w)) else w
        low = w.lower()
        for cand in (w, low, trim_elongation(low), _DEDUP.sub(r"\1", low)):
            if cand in norm_map: return norm_map[cand]
        return trim_elongation(low) if (w.isalpha() or is_kn(w)) else w

    def simple(fn, tags=None):
        def f(text):
            t, k = process_comment(text)
            return " ".join(fn(w) if (tags is None or kk in tags) else w
                            for w, kk in zip(t, k))
        return f

    def elected(keyfn, guarded):
        b = defaultdict(Counter)
        for w, n in TRAIN_FREQ.items():
            kk = keyfn(w)
            if kk: b[kk][w] += n
        m = {w: c.most_common(1)[0][0] for kk, c in b.items() for w in c}
        if guarded:
            vet = Counter(); out = {}
            for w, c in m.items():
                g = veto(w, c) if w != c else None
                if g: out[w] = w; vet[g] += 1
                else: out[w] = c
            print(f"  GUARD: vetoed {sum(vet.values()):,} of {len(m):,} merges {dict(vet)}")
            m = out
        # Must consume the SAME token stream as every other arm. Splitting the
        # raw string on whitespace keeps emoji clusters ("😥🙏🙏❤️") as one
        # token while process_comment() splits them into separate graphemes,
        # which is what produced the token-count mismatches.
        def fn(text):
            toks, tags = process_comment(text)
            return " ".join(w if k in PASSTHROUGH else m.get(w.lower(), w)
                            for w, k in zip(toks, tags))
        return fn

    lccn_obj = {}
    A = {}
    for name in which:
        if   name == "raw":               A[name] = lambda t: " ".join(process_comment(t)[0])
        elif name == "indicnlp":          A[name] = simple(_indicnlp, {"kannada"})
        elif name in ("indicxlit", "itrans"):
            if name not in XL: print(f"  skip {name}: backend unavailable"); continue
            A[name] = simple(_tl(name), {"romanized_kannada"})
        elif name == "aksharamukha":      A[name] = simple(_aksh, {"kannada"})
        elif name == "keel":              A[name] = simple(keel_lookup)
        elif name == "aksharamukha_keel": A[name] = simple(lambda w: _aksh(keel_lookup(w)))
        elif name in ("keel_indicxlit", "keel_itrans"):
            bk = name.split("keel_")[1]
            if bk not in XL: print(f"  skip {name}: backend unavailable"); continue
            def _mk(bk):
                tl = _tl(bk)
                def f(text):
                    t, k = process_comment(text)
                    return " ".join(tl(keel_lookup(w)) if kk == "romanized_kannada"
                                    else keel_lookup(w) for w, kk in zip(t, k))
                return f
            A[name] = _mk(bk)
        elif name == "kope":              A[name] = simple(lambda w: kope(w) if kope_ok(w) else w)
        elif name == "terun":             A[name] = simple(lambda w: (terun(w) or w) if kope_ok(w) else w)
        elif name in ("soundex","soundex_guard"):
            A[name] = elected(soundex, name.endswith("guard"))
        elif name in ("indicsoundex","indicsoundex_guard"):
            A[name] = elected(indic_soundex, name.endswith("guard"))
        elif name == "metaphone":         A[name] = elected(metaphone, False)
        elif name == "lccn_base":
            # original lineage: the three vetoes only, no G-test, no cluster
            # coherence, no orthographic guard. Kept so the paper can report
            # what each added guard contributes.
            o = LCCN(norm_map, evidence, veto, use_gtest=False,
                     use_cluster=False, use_ortho=False, lex=LEX)
            lccn_obj[name] = o; A[name] = o.normalize
        elif name == "lccn":
            o = LCCN(norm_map, evidence, veto, lex=LEX); lccn_obj[name] = o; A[name] = o.normalize
        elif name in ("lccn_indicxlit", "lccn_itrans"):
            bk = name.split("lccn_")[1]
            if bk not in XL: print(f"  skip {name}: backend unavailable"); continue
            o = LCCN(norm_map, evidence, veto, translit=_tl(bk), lex=LEX)
            lccn_obj[name] = o; A[name] = o.normalize
        elif name in ("lccn_base_indicxlit", "lccn_base_itrans"):
            bk = name.split("lccn_base_")[1]
            if bk not in XL: print(f"  skip {name}: backend unavailable"); continue
            o = LCCN(norm_map, evidence, veto, translit=_tl(bk), use_gtest=False,
                     use_cluster=False, use_ortho=False)
            lccn_obj[name] = o; A[name] = o.normalize
        else: raise ValueError(f"unknown arm: {name}")
    return A, lccn_obj, veto

def xlit_arms(names, eng=None):
    """Expands any transliteration-bearing arm name into one arm per available
    backend, so a single run can carry both itrans and indicxlit columns. The
    column name always states which engine produced it."""
    backends = list(eng) if eng else ["itrans"]
    out = []
    for n in names:
        if "indicxlit" in n or "itrans" in n:
            stem = n.replace("indicxlit", "{x}").replace("itrans", "{x}")
            out += [stem.format(x=b) for b in backends]
        else:
            out.append(n)
    seen = set()
    return [a for a in out if not (a in seen or seen.add(a))]

def change_rates(df, arms, raw_col="text_raw"):
    rows = []
    for name in arms:
        if name == "raw": continue
        a, b = df[raw_col], df[f"text_{name}"]
        cc = int((a != b).sum()); ct = tt = 0
        for x, y in zip(a, b):
            rt, at = x.split(), y.split(); tt += len(rt)
            if len(rt) == len(at): ct += sum(1 for p, q in zip(rt, at) if p != q)
            else: ct += sum((Counter(rt) - Counter(at)).values())
        rows.append({"method": name, "comment_change_%": round(100*cc/len(df), 2),
                     "token_change_%": round(100*ct/max(tt,1), 2)})
    return pd.DataFrame(rows)

def lccn_ablation(norm_map, F2E, evidence, frame, TXT, sample_n=4000, lex=None):
    """Adds one guard at a time and reports what each blocks. Produces the
    ablation table for the LCCN paper."""
    veto = make_veto(F2E, lex)
    steps = [
        ("no guards",            dict(use_veto=False, use_gtest=False, use_cluster=False, use_ortho=False)),
        ("+ three vetoes",       dict(use_veto=True,  use_gtest=False, use_cluster=False, use_ortho=False)),
        ("+ orthographic guard", dict(use_veto=True,  use_gtest=False, use_cluster=False, use_ortho=True)),
        ("+ G-test",             dict(use_veto=True,  use_gtest=True,  use_cluster=False, use_ortho=True)),
        ("+ cluster coherence",  dict(use_veto=True,  use_gtest=True,  use_cluster=True,  use_ortho=True)),
    ]
    s = frame.sample(min(sample_n, len(frame)), random_state=SEED)
    rows = []
    for name, kw in steps:
        o = LCCN(norm_map, evidence, veto, lex=lex, **kw)
        ch = tot = 0
        for txt in s[TXT]:
            a = " ".join(process_comment(txt)[0]).lower().split()
            b = o.normalize(txt).lower().split()
            tot += len(a); ch += sum(1 for x, y in zip(a, b) if x != y)
        st = o.stats
        rows.append({"variant": name,
                     "recovery_%": round(100*ch/max(tot,1), 2),
                     "merged": st["merged"],
                     "blocked_veto": sum(v for k, v in st.items() if k.startswith("blocked_veto")),
                     "blocked_ortho": st["blocked_orthographic"],
                     "blocked_gtest": st["blocked_gtest"],
                     "blocked_cluster": st["blocked_cluster"]})
        print(f"  {name:<22} recovery {rows[-1]['recovery_%']:6.2f}%  merged {st['merged']:,}")
    return pd.DataFrame(rows)

def fingerprint():
    h = hashlib.sha256(open(__file__, "rb").read()).hexdigest()[:16]
    print(f"pipeline v{PIPELINE_VERSION}  sha256:{h}  SEGMENT={SEGMENT}")
    return h


Writing pipeline.py


In [4]:
# ================================================================
#  CELL 2 — SHARED SETUP.  IDENTICAL IN BOTH NOTEBOOKS.
#  Corpus audit, KEEL map, id split, train-only evidence, xlit cache.
#  Must print "pipeline v5.3". If it prints anything else, CELL 1 did
#  not run and pipeline.py on disk is stale.
# ================================================================
import importlib, pipeline
importlib.reload(pipeline)
from pipeline import *
import pandas as pd, numpy as np, joblib
from collections import Counter, defaultdict

FP = pipeline.fingerprint()
assert pipeline.PIPELINE_VERSION == "5.3", \
    f"stale pipeline.py (v{pipeline.PIPELINE_VERSION}). Re-run CELL 1, then Restart session."

df, ID, TXT, LAB, RATERS = load_corpus()

norm_map, F2E, LEX = build_keel()
joblib.dump(norm_map, f"{WORK}/keel_norm_map.pkl")

try:
    from indicnlp import common
    import glob as _g
    for c in _g.glob("/kaggle/input/**/indic_nlp_resources*", recursive=True):
        common.set_resources_path(c); break
    from indicnlp.normalize.indic_normalize import IndicNormalizerFactory
    _indic_norm = IndicNormalizerFactory().get_normalizer("kn")
    _ = _indic_norm.normalize("ಪರೀಕ್ಷೆ"); print("IndicNLP: FULL")
except Exception as e:
    import unicodedata
    class _F:
        def normalize(self, s): return unicodedata.normalize("NFC", s)
    _indic_norm = _F(); print("IndicNLP: FALLBACK(NFC) —", str(e)[:60])
pipeline._indic_norm = _indic_norm

df["text_raw"] = df[TXT].map(lambda t: " ".join(process_comment(t)[0]))

# ---- SPLIT FIRST, keyed on the id column ----
SPLIT, TEST_HASH = make_split(df, ID, LAB)
train = df[df[ID].isin(SPLIT["train"])]

# ---- label evidence and phonetic election: TRAIN ONLY ----
evidence = LabelEvidence(train, TXT, LAB)
TRAIN_FREQ = Counter()
for t in train.text_raw: TRAIN_FREQ.update(w.lower() for w in t.split())
pipeline.TRAIN_FREQ = TRAIN_FREQ
print(f"train vocab {len(TRAIN_FREQ):,} | evidence over {len(evidence.counts):,} forms")

# ---- IndicXlit from the cache. Writes romanized_vocab.csv and stops if
#      no xlit_cache.csv is attached. Never falls back to sanscript. ----
xlit = load_xlit(df, TXT)

if xlit is None:
    print("\nSTOPPED HERE — build the transliteration cache, then rerun.")
    print("Everything above completed: corpus, KEEL map, split, evidence.")
else:
    print("\nSHARED SETUP DONE — the next cell is the only paper-specific one.")


pipeline v5.3  sha256:6a536dd437ba85b0  SEGMENT=True
corpus: 40,000 -> 40,000 after audit (dedup + conflict removal)
majority
disgust    10000
happy      10000
sad        10000
anger      10000
  KEEL cascade | tier1 126,567 | tier2 0 | tier3 603 | emotions 174,425 (3,658 ambiguous)
    multi-word entries skipped (word-level lookup): 274,777
    negated phrases kept for the context-aware veto: 706 | va_id misses 0
  KEEL entries: 126,567 | seg vocab: 12,326 | form->emotion: 174,425
IndicNLP: FULL
train  28,000
val     6,000
test    6,000
leakage check passed | test-set id hash: e2d6e1f7d564cace
train vocab 42,985 | evidence over 42,990 forms
transliteration backend: ITRANS (indic_transliteration.sanscript)
converted 26,391 of 26,406 romanized types to Kannada script (99.9%)
IndicXlit cache: 26,742 entries (99.9% to Kannada script)
cache covers 26,399 of 26,406 romanized types (100.0%)

transliteration backends ready: ['itrans', 'indicxlit']

SHARED SETUP DONE — the next cell is the onl

In [5]:
# ================================================================
#  PAPER 1 (LCCN) — the ONLY paper-specific cell.
#
#  WANTED lists every arm the paper needs, both transliteration backends
#  written out explicitly. xlit_arms() keeps only those whose backend is
#  actually loaded, and the check below says plainly which are missing
#  instead of silently producing a shorter list.
#
#  full set = 13 arms:
#    raw, keel, kope, metaphone, terun,
#    soundex, soundex_guard, indicsoundex, indicsoundex_guard,
#    lccn_base, lccn, lccn_itrans, lccn_indicxlit
# ================================================================
WANTED = ["raw", "keel", "kope", "metaphone", "terun",
          "soundex", "soundex_guard", "indicsoundex", "indicsoundex_guard",
          "lccn_base", "lccn", "lccn_itrans", "lccn_indicxlit"]

available = set(xlit) if isinstance(xlit, dict) else set()
WHICH = [a for a in WANTED
         if not ("itrans" in a or "indicxlit" in a)
         or ("indicxlit" in a and "indicxlit" in available)
         or ("itrans" in a and "indicxlit" not in a and "itrans" in available)]
TAG = "paper1_lccn"
RUN_ABLATION = True

print(f"arms for this run ({len(WHICH)}):", WHICH)
missing = [a for a in WANTED if a not in WHICH]
if missing:
    print("\n" + "!"*66)
    print("MISSING ARMS:", missing)
    print("The indicxlit backend was not loaded. Fix ONE of these:")
    print("  a) Add Input -> Notebook Output -> the run that made xlit_cache.csv")
    print("  b) run this notebook on the Python 3.10 image (Copy & Edit the")
    print("     script-conversion notebook) so the cache can be built live")
    print("Check cell 3 for the line: transliteration backends ready: [...]")
    print("!"*66)


arms for this run (13): ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit']


In [6]:
# ================================================================
#  CELL 5 — APPLY ARMS, CHANGE RATES, SAVE.  IDENTICAL IN BOTH.
#  Skips cleanly (no error) if the transliteration cache is not attached,
#  so Save & Run All completes and romanized_vocab.csv reaches the Output tab.
# ================================================================
if xlit is None:
    print("SKIPPED — no xlit_cache.csv attached.")
    print("Download romanized_vocab.csv from the Output tab, build the cache")
    print("with build_xlit_cache.py, attach it as a dataset, and rerun.")
else:
    ARMS, LCCN_OBJ, veto = build_arms(norm_map, F2E, evidence, xlit, WHICH, LEX=LEX)
    print("arms:", list(ARMS), "\n")

    from tqdm.auto import tqdm; tqdm.pandas()
    for name, fn in ARMS.items():
        df[f"text_{name}"] = df[TXT].progress_apply(fn); print("built text_" + name)

    # ---- consistency: every arm must emit the same token count as raw ----
    bad = 0
    for s in df[TXT].sample(min(200, len(df)), random_state=SEED):
        ref = len(ARMS["raw"](s).split())
        for n, f in ARMS.items():
            if len(f(s).split()) != ref:
                bad += 1
                if bad <= 5: print(f"  MISMATCH {n}: {s[:50]}")
    print("all arms consistent" if bad == 0 else f"{bad} token-count mismatches")

    for n, o in LCCN_OBJ.items(): o.report(n)
    LEX.report()

    cr = change_rates(df, ARMS)
    print("\n=== CHANGE RATE (vs raw) ===")
    print(cr.to_string(index=False))
    cr.to_csv(f"{WORK}/change_rates_{TAG}.csv", index=False)

    cols = [ID, TXT, LAB, "text_raw"] + [f"text_{a}" for a in ARMS if a != "raw"]
    for n in ("train", "val", "test"):
        sub = df[df[ID].isin(SPLIT[n])][cols].sort_values(ID)
        sub.to_csv(f"{WORK}/splits/{n}.csv", index=False, encoding="utf-8-sig")
        print(f"{n:<6}{len(sub):>7,}")
    print(f"\npipeline {FP} | test-set id hash {TEST_HASH} | SEGMENT={SEGMENT}")

    if RUN_ABLATION:
        print("\n=== LCCN GUARD ABLATION ===")
        ab = lccn_ablation(norm_map, F2E, evidence, train, TXT, lex=LEX)
        ab.to_csv(f"{WORK}/lccn_ablation_{TAG}.csv", index=False)
        print(); print(ab.to_string(index=False))

    print("DONE — Save Version, then add as a dataset for Notebook 2.")


  GUARD: vetoed 3,280 of 26,509 merges {'length': 2230, 'emotion': 845, 'negation': 205}
  GUARD: vetoed 3,326 of 26,509 merges {'length': 2252, 'emotion': 873, 'negation': 201}
arms: ['raw', 'keel', 'kope', 'metaphone', 'terun', 'soundex', 'soundex_guard', 'indicsoundex', 'indicsoundex_guard', 'lccn_base', 'lccn', 'lccn_itrans', 'lccn_indicxlit'] 



  0%|          | 0/40000 [00:00<?, ?it/s]

built text_raw


  0%|          | 0/40000 [00:00<?, ?it/s]

built text_keel


...

built text_lccn


  0%|          | 0/40000 [00:00<?, ?it/s]

built text_lccn_itrans


  0%|          | 0/40000 [00:00<?, ?it/s]

built text_lccn_indicxlit
all arms consistent
  lccn_base: merged 8,058 of 54,694 distinct forms
    blocked_veto_emotion             197
    blocked_veto_length                5
    blocked_veto_negation             49
  lccn: merged 5,977 of 54,694 distinct forms
    blocked_cluster                   35
    blocked_gtest                    259
    blocked_orthographic           4,062
    blocked_veto_emotion             114
    blocked_veto_length                3
    blocked_veto_negation             44
  lccn_itrans: merged 5,977 of 54,694 distinct forms
    blocked_cluster                   35
    blocked_gtest                    259
    blocked_orthographic           4,062
    blocked_veto_emotion             114
    blocked_veto_length                3
    blocked_veto_negation             44
  lccn_indicxlit: merged 5,977 of 54,694 distinct forms
    blocked_cluster                   35
    blocked_gtest                    259
    blocked_orthographic           4,062
    blocke